# Inter-Annotator Agreement (IAA): Reva vs Ryan

This notebook computes agreement between two annotators across two multi-label tasks:

1. **Target Label Mapping** (`reva_labels.tsv` vs `ryan_labels.tsv`)
2. **Dogwhistle Inferred Target** (`reva_glossary.tsv` vs `ryan_glossary.tsv`)

For both tasks, each item can contain 0 to N labels represented as a comma-separated string.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from tabulate import tabulate
from IPython.display import Markdown, display

In [ ]:
# Annotation TSV files live in the same directory as this notebook.
ANNOTATIONS_DIR = Path.cwd()

def parse_labels(value) -> frozenset:
    """Parse comma-separated label string to frozenset of stripped labels.
    Returns frozenset() for NaN or empty."""
    if pd.isna(value) or str(value).strip() == "":
        return frozenset()
    return frozenset(lbl.strip() for lbl in str(value).split(",") if lbl.strip())


def jaccard_similarity(a: frozenset, b: frozenset) -> float:
    if not a and not b:
        return 1.0
    union = a | b
    if not union:
        return 1.0
    return len(a & b) / len(union)


def krippendorff_alpha_jaccard(a_sets: list, b_sets: list) -> float:
    observed_disagreement = np.mean([1.0 - jaccard_similarity(a, b) for a, b in zip(a_sets, b_sets)])

    pooled = list(a_sets) + list(b_sets)
    m = len(pooled)
    if m < 2:
        return np.nan

    pairwise_sum_unordered = 0.0
    for i in range(m):
        for j in range(i + 1, m):
            pairwise_sum_unordered += 1.0 - jaccard_similarity(pooled[i], pooled[j])

    # Convert unordered-pair sum to ordered-pair mean over i != j.
    expected_disagreement = (2.0 * pairwise_sum_unordered) / (m * (m - 1))
    if np.isclose(expected_disagreement, 0.0):
        return 1.0

    return 1.0 - (observed_disagreement / expected_disagreement)


def load_task(
    reva_path: Path,
    ryan_path: Path,
    join_keys: list,
    label_col: str,
) -> pd.DataFrame:
    reva = pd.read_csv(reva_path, sep="\t", dtype=str).rename(columns={label_col: f"{label_col}_reva"})
    ryan = pd.read_csv(ryan_path, sep="\t", dtype=str).rename(columns={label_col: f"{label_col}_ryan"})

    merged = reva.merge(ryan, on=join_keys, how="inner")
    merged[f"{label_col}_reva"] = merged[f"{label_col}_reva"].apply(parse_labels)
    merged[f"{label_col}_ryan"] = merged[f"{label_col}_ryan"].apply(parse_labels)
    return merged


def compute_metrics(df: pd.DataFrame, label_col: str):
    a_sets = df[f"{label_col}_reva"].tolist()
    b_sets = df[f"{label_col}_ryan"].tolist()

    exact_match = np.mean([a == b for a, b in zip(a_sets, b_sets)])
    mean_jaccard = np.mean([jaccard_similarity(a, b) for a, b in zip(a_sets, b_sets)])

    all_labels = sorted(set().union(*a_sets).union(*b_sets))
    kappa_rows = []
    kappas = []

    for label in all_labels:
        a_vec = [1 if label in s else 0 for s in a_sets]
        b_vec = [1 if label in s else 0 for s in b_sets]

        # Skip labels that are all-zero for both annotators.
        if sum(a_vec) == 0 and sum(b_vec) == 0:
            continue

        kappa = cohen_kappa_score(a_vec, b_vec)
        if pd.isna(kappa):
            continue

        kappas.append(kappa)
        kappa_rows.append({
            "label": label,
            "kappa": float(kappa),
            "pct_reva_assigned": 100.0 * np.mean(a_vec),
            "pct_ryan_assigned": 100.0 * np.mean(b_vec),
        })

    breakdown = pd.DataFrame(kappa_rows)
    if not breakdown.empty:
        breakdown["abs_kappa"] = breakdown["kappa"].abs()
        breakdown = breakdown.sort_values("abs_kappa", ascending=False).reset_index(drop=True)

    result = {
        "n_items": len(df),
        "exact_match_pct": 100.0 * float(exact_match),
        "mean_jaccard": float(mean_jaccard),
        "macro_cohen_kappa": float(np.mean(kappas)) if kappas else np.nan,
        "krippendorff_alpha_jaccard": float(krippendorff_alpha_jaccard(a_sets, b_sets)),
        "n_labels_for_kappa": len(kappas),
    }
    return result, breakdown

In [ ]:
labels_df = load_task(
    reva_path=ANNOTATIONS_DIR / "reva_labels.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_labels.tsv",
    join_keys=["dataset", "raw_target_label"],
    label_col="dest_label",
)

glossary_df = load_task(
    reva_path=ANNOTATIONS_DIR / "reva_glossary.tsv",
    ryan_path=ANNOTATIONS_DIR / "ryan_glossary.tsv",
    join_keys=["term"],
    label_col="inferred_target",
)

labels_metrics, labels_breakdown = compute_metrics(labels_df, "dest_label")
glossary_metrics, glossary_breakdown = compute_metrics(glossary_df, "inferred_target")

print(f"Labels items:   {labels_metrics['n_items']}")
print(f"Glossary items: {glossary_metrics['n_items']}")

## What each metric means (plain language)

- **Exact Match %**: how often Reva and Ryan picked exactly the same set of labels for an item.
- **Mean Jaccard Similarity**: average overlap between their label sets; partial overlap gets partial credit.
- **Macro-Averaged Cohen's κ**: for each label, treat annotation as yes/no and compute κ; then average those κ values across labels.
- **Krippendorff's α (Jaccard distance)**: agreement adjusted for chance using set-distance disagreement; implemented directly from the formula without external α packages.

In [ ]:
summary = pd.DataFrame([
    {"Metric": "N Items", "Labels Task": labels_metrics["n_items"], "Glossary Task": glossary_metrics["n_items"]},
    {"Metric": "Exact Match %", "Labels Task": f"{labels_metrics['exact_match_pct']:.2f}%", "Glossary Task": f"{glossary_metrics['exact_match_pct']:.2f}%"},
    {"Metric": "Mean Jaccard Similarity", "Labels Task": f"{labels_metrics['mean_jaccard']:.3f}", "Glossary Task": f"{glossary_metrics['mean_jaccard']:.3f}"},
    {"Metric": "Macro-Avg Cohen's κ", "Labels Task": f"{labels_metrics['macro_cohen_kappa']:.3f} (N={labels_metrics['n_labels_for_kappa']} lbl)", "Glossary Task": f"{glossary_metrics['macro_cohen_kappa']:.3f} (N={glossary_metrics['n_labels_for_kappa']} lbl)"},
    {"Metric": "Krippendorff's α (Jaccard dist)", "Labels Task": f"{labels_metrics['krippendorff_alpha_jaccard']:.3f}", "Glossary Task": f"{glossary_metrics['krippendorff_alpha_jaccard']:.3f}"},
])

def stripe_rows(row):
    color = "#f8fbff" if row.name % 2 == 0 else "#eef5fb"
    return [f"background-color: {color}"] * len(row)

styled_summary = (
    summary.style
    .set_caption("Table X: Inter-Annotator Agreement (IAA) across two annotation tasks.")
    .apply(stripe_rows, axis=1)
    .set_properties(subset=["Metric"], **{"font-weight": "bold"})
    .set_table_styles([
        {"selector": "th", "props": [("background-color", "#dbe9f4"), ("font-weight", "bold")]},
        {"selector": "caption", "props": [("caption-side", "top"), ("font-size", "1.05em"), ("font-weight", "bold"), ("padding", "6px")]},
    ])
    .hide(axis="index")
)

display(styled_summary)
display(Markdown(
    "κ = macro-averaged Cohen's kappa across all binary per-label vectors. "
    "α computed with Jaccard distance metric. "
    "N labels = number of distinct labels with non-trivial variance used for κ computation."
))

print(tabulate(summary, headers="keys", tablefmt="github", showindex=False))

In [ ]:
def format_kappa_breakdown(breakdown: pd.DataFrame, task_name: str, top_n: int = 20) -> pd.DataFrame:
    if breakdown.empty:
        return pd.DataFrame(columns=["label", "κ", "% Reva assigned", "% Ryan assigned"])

    top = breakdown.sort_values("abs_kappa", ascending=False).head(top_n).copy()
    out = top[["label", "kappa", "pct_reva_assigned", "pct_ryan_assigned"]].rename(columns={
        "kappa": "κ",
        "pct_reva_assigned": "% Reva assigned",
        "pct_ryan_assigned": "% Ryan assigned",
    })
    out["κ"] = out["κ"].map(lambda x: f"{x:.3f}")
    out["% Reva assigned"] = out["% Reva assigned"].map(lambda x: f"{x:.2f}%")
    out["% Ryan assigned"] = out["% Ryan assigned"].map(lambda x: f"{x:.2f}%")
    out.insert(0, "Task", task_name)
    return out

labels_top20 = format_kappa_breakdown(labels_breakdown, "Labels Task", top_n=20)
glossary_top20 = format_kappa_breakdown(glossary_breakdown, "Glossary Task", top_n=20)

display(Markdown("### Per-label κ breakdown (Top-20 by |κ|): Labels Task"))
display(labels_top20.style.hide(axis="index").set_properties(subset=["label"], **{"font-weight": "bold"}))

display(Markdown("### Per-label κ breakdown (Top-20 by |κ|): Glossary Task"))
display(glossary_top20.style.hide(axis="index").set_properties(subset=["label"], **{"font-weight": "bold"}))